In [ ]:
# -*- coding: utf-8 -*-
# 1. ИМПОРТ НЕОБХОДИМЫХ БИБЛИОТЕК
import numpy as np                # работа с массивами и математикой
import pandas as pd               # сохранение данных в таблицы (CSV)
import matplotlib.pyplot as plt   # рисование графиков
from sklearn.model_selection import train_test_split  # разделение данных на обучение и тест
from sklearn.preprocessing import StandardScaler      # масштабирование данных
from sklearn.metrics import r2_score, mean_absolute_error  # метрики качества

import torch                      # библиотека глубокого обучения PyTorch
import torch.nn as nn             # модуль для построения нейронных сетей
import torch.optim as optim       # оптимизаторы (алгоритмы настройки весов)
from torch.utils.data import DataLoader, TensorDataset  # удобная загрузка данных

In [ ]:
# Фиксируем случайные числа, чтобы результаты были воспроизводимы
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# 2. ФУНКЦИЯ, ВЫЧИСЛЯЮЩАЯ ТЕОРЕТИЧЕСКУЮ КРИВУЮ ПОЛЗУЧЕСТИ
def creep_strain(t, a0, a1, a2):
    """
    Возвращает значение деформации ε в момент времени t
    по формуле ε(t) = a0 + a1 * exp(-a2 * t).
    Параметры:
        t   : число или массив времён
        a0, a1, a2 : параметры модели
    """
    return a0 + a1 * np.exp(-a2 * t)


In [ ]:
# 3. ГЕНЕРАЦИЯ ДАННЫХ С ФИЗИЧЕСКИМИ ОГРАНИЧЕНИЯМИ

def generate_dataset_physical(n_samples=100, t_points=None, noise_level=0.01):
    """
    Генерирует синтетические кривые ползучести, гарантируя:
      - ε(t) > 0 для всех t
      - начальное значение ε(0) > 0.1
      - форма кривой: быстрый рост и выход на плато
    """
    if t_points is None:
        t_points = np.array([0.1, 0.3, 0.5, 1.0, 2.0])

    X_list, y_list = [], []
    max_attempts = 1000  # чтобы не зациклиться

    for _ in range(n_samples):
        for attempt in range(max_attempts):
            # Сужаем диапазоны для реалистичных кривых
            a0 = np.random.uniform(0.6, 1.5)       # плато
            a1 = np.random.uniform(-0.8, -0.2)     # отрицательный, но не слишком большой
            a2 = np.random.uniform(2.0, 8.0)       # достаточно быстрое затухание

            # Проверка: начальная деформация должна быть положительной
            if a0 + a1 <= 0.05:
                continue

            # Проверяем, что на всех временных точках деформация > 0
            eps_vals = creep_strain(t_points, a0, a1, a2)
            if np.any(eps_vals <= 0):
                continue

            # Успешно – выходим из цикла попыток
            break
        else:
            # Если не удалось подобрать за max_attempts, берём последние значения
            pass

        # Добавляем шум
        noise = noise_level * np.random.randn(len(t_points)) * np.abs(eps_vals)
        eps_noisy = eps_vals + noise
        eps_noisy = np.maximum(eps_noisy, 1e-6)  # гарантируем неотрицательность

        X_list.append(eps_noisy)
        y_list.append([a0, a1, a2])

    X = np.vstack(X_list)
    y = np.vstack(y_list)
    return X, y, t_points

print(">>> Генерация физически корректных данных...")
t_grid = np.array([0.1, 0.3, 0.5, 1.0, 2.0])
X, y, t_grid = generate_dataset_physical(n_samples=100, noise_level=0.01)
print(f"    Создано {X.shape[0]} образцов.")

# Сохраняем в CSV
df = pd.DataFrame(X, columns=[f"eps_t{t}" for t in t_grid])
df[['a0', 'a1', 'a2']] = y
df.to_csv("creep_dataset_physical.csv", index=False)
print(">>> Данные сохранены в 'creep_dataset_physical.csv'.")

t_fine = np.linspace(0, 2, 200)
plt.figure(figsize=(12, 6))
for i in range(3):
    a0, a1, a2 = y[i]
    eps_smooth = creep_strain(t_fine, a0, a1, a2)
    plt.plot(t_fine, eps_smooth, '-', linewidth=1.5, label=f'Образец {i+1}')
    plt.plot(t_grid, X[i], 'o', markersize=6)
plt.xlabel('Время t, с')
plt.ylabel('Деформация ε')
plt.title('Примеры сгенерированных кривых ползучести')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Дополнительно покажем одну кривую на более мелкой сетке для проверки формы
#t_fine = np.linspace(0, 2, 200)
#sample_params = y[0]
#eps_fine = creep_strain(t_fine, *sample_params)
#plt.figure(figsize=(9, 5))
#plt.plot(t_fine, eps_fine, 'b-', label='Идеальная кривая (без шума)')
#plt.plot(t_grid, X[0], 'ro', markersize=8, label='Измерения с шумом')
#plt.xlabel('Время t, с')
#plt.ylabel('Деформация ε')
#plt.title('Форма кривой ползучести')
#plt.legend()
#plt.grid(True)
#plt.show()

In [ ]:
# 4. ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ НЕЙРОСЕТИ
# Разделяем данные: 80% для обучения, 20% для итоговой проверки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# Масштабирование (очень важно для нейросетей!)
# Мы вычитаем среднее и делим на стандартное отклонение, чтобы все признаки
# имели примерно одинаковый масштаб. Это ускоряет и стабилизирует обучение.
scaler_X = StandardScaler()
scaler_y = StandardScaler()

# Обучаем scaler ТОЛЬКО на тренировочных данных, затем применяем к тестовым
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)

# Переводим данные в тензоры PyTorch (формат, понятный библиотеке)
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t  = torch.tensor(y_test_scaled, dtype=torch.float32)

# DataLoader позволяет подавать данные порциями (batch) во время обучения
batch_size = 16
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# 5. АРХИТЕКТУРА НЕЙРОННОЙ СЕТИ
class CreepParameterNet(nn.Module):
    """
    Полносвязная нейронная сеть (многослойный перцептрон).
    На вход подаётся 5 чисел (значения ε в 5 моментов времени),
    на выходе – 3 числа (a0, a1, a2).
    """
    def __init__(self, input_dim, hidden_sizes=[32, 16, 8]):
        super(CreepParameterNet, self).__init__()
        layers = []
        prev_size = input_dim
        for h_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, h_size))  # полносвязный слой
            layers.append(nn.ReLU())                     # функция активации ReLU
            prev_size = h_size
        layers.append(nn.Linear(prev_size, 3))           # выходной слой (3 параметра)
        self.net = nn.Sequential(*layers)                # объединяем всё в одну цепочку

    def forward(self, x):
        """Прямой проход данных через сеть."""
        return self.net(x)

# Создаём экземпляр сети
input_dim = X_train.shape[1]   # 5
model = CreepParameterNet(input_dim)
print("\n>>> Архитектура нейросети:")
print(model)

In [ ]:
# 6. НАСТРОЙКА ОБУЧЕНИЯ
criterion = nn.MSELoss()                # функция потерь – среднеквадратичная ошибка
optimizer = optim.Adam(model.parameters(), lr=0.001)  # оптимизатор Adam

epochs = 800                             # сколько раз пройти по всем данным
train_loss_history = []
test_loss_history = []

print("\n>>> Начало обучения...")
for epoch in range(epochs):
    # ---- Фаза обучения ----
    model.train()
    total_train_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()            # обнуляем градиенты
        predictions = model(batch_X)     # прямой проход
        loss = criterion(predictions, batch_y)  # вычисляем ошибку
        loss.backward()                  # обратный проход (градиенты)
        optimizer.step()                 # обновляем веса
        total_train_loss += loss.item() * batch_X.size(0)

    avg_train_loss = total_train_loss / len(train_loader.dataset)
    train_loss_history.append(avg_train_loss)

    # ---- Оценка на тестовой выборке (без обновления весов) ----
    model.eval()
    with torch.no_grad():
        test_pred = model(X_test_t)
        test_loss = criterion(test_pred, y_test_t).item()
        test_loss_history.append(test_loss)

    if (epoch + 1) % 100 == 0:
        print(f"Эпоха {epoch+1:3d}/{epochs} | Train Loss: {avg_train_loss:.6f} | Test Loss: {test_loss:.6f}")

# Рисуем график изменения ошибки в процессе обучения
plt.figure(figsize=(8, 5))
plt.plot(train_loss_history, label='Ошибка на обучении')
plt.plot(test_loss_history, label='Ошибка на тесте')
plt.xlabel('Эпоха')
plt.ylabel('MSE Loss')
plt.title('Динамика обучения нейросети')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# 7. ТЕСТИРОВАНИЕ И ОЦЕНКА КАЧЕСТВА
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_t).numpy()
    # Возвращаем предсказания к исходному масштабу
    y_pred = scaler_y.inverse_transform(y_pred_scaled)
    y_true = y_test

# Вычисляем основные метрики
mse = np.mean((y_true - y_pred) ** 2)
r2 = r2_score(y_true, y_pred, multioutput='uniform_average')
mae = mean_absolute_error(y_true, y_pred)

print("\n>>> РЕЗУЛЬТАТЫ НА ТЕСТОВОЙ ВЫБОРКЕ (20% данных, которые сеть не видела при обучении):")
print(f"    Среднеквадратичная ошибка (MSE) : {mse:.6f}")
print(f"    Коэффициент детерминации R²   : {r2:.4f}  (чем ближе к 1, тем лучше)")
print(f"    Средняя абсолютная ошибка (MAE): {mae:.6f}")

# Оцениваем качество отдельно по каждому параметру
param_names = ['a0', 'a1', 'a2']
for i, name in enumerate(param_names):
    r2_i = r2_score(y_true[:, i], y_pred[:, i])
    mae_i = mean_absolute_error(y_true[:, i], y_pred[:, i])
    print(f"    {name}: R² = {r2_i:.4f}, MAE = {mae_i:.4f}")

# Строим диаграммы рассеяния "Истина vs Предсказание"
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, name in enumerate(param_names):
    ax = axes[i]
    ax.scatter(y_true[:, i], y_pred[:, i], alpha=0.7)
    # Рисуем линию идеального совпадения
    min_val = min(y_true[:, i].min(), y_pred[:, i].min())
    max_val = max(y_true[:, i].max(), y_pred[:, i].max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--')
    ax.set_xlabel(f'Истинное значение {name}')
    ax.set_ylabel(f'Предсказанное значение {name}')
    ax.set_title(f'Параметр {name}')
    ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 8. ПРИМЕР ВОССТАНОВЛЕНИЯ ПОЛНОЙ КРИВОЙ ПОЛЗУЧЕСТИ
# Возьмём один образец из тестовой выборки и сравним истинную кривую,
# кривую по предсказанным параметрам и зашумлённые точки, поданные на вход.
sample_idx = 0   # можно изменить, чтобы посмотреть другие образцы
true_params = y_true[sample_idx]
pred_params = y_pred[sample_idx]

print("\n>>> Пример для одного тестового образца:")
print(f"    Истинные параметры:     a0 = {true_params[0]:.3f}, a1 = {true_params[1]:.3f}, a2 = {true_params[2]:.3f}")
print(f"    Предсказанные параметры: a0 = {pred_params[0]:.3f}, a1 = {pred_params[1]:.3f}, a2 = {pred_params[2]:.3f}")

# Строим гладкие кривые на более мелкой сетке времени
t_fine = np.linspace(0.1, 2.0, 200)
eps_true_curve = creep_strain(t_fine, *true_params)
eps_pred_curve = creep_strain(t_fine, *pred_params)
# Точки, которые сеть видела на входе (с шумом)
eps_observed = X_test[sample_idx]

plt.figure(figsize=(9, 5))
plt.plot(t_fine, eps_true_curve, 'b-', linewidth=2, label='Истинная кривая')
plt.plot(t_fine, eps_pred_curve, 'r--', linewidth=2, label='Кривая по предсказанным параметрам')
plt.plot(t_grid, eps_observed, 'ko', markersize=8, label='Входные данные (с шумом)')
plt.xlabel('Время t, с')
plt.ylabel('Деформация ε')
plt.title('Сравнение истинной и восстановленной кривых ползучести')
plt.legend()
plt.grid(True)
plt.show()